## 1. Celda 1: Configuración Inicial

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, avg, sum as spark_sum, count, max as spark_max, round, desc
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
import os
import shutil
from datetime import datetime

print("=" * 60)
print(f"TALLER EVALUATIVO 1 - INICIO: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

spark = (SparkSession.builder
         .appName("EvalTaller1")
         .master("local[*]")
         .config("spark.driver.memory", "1g")
         .getOrCreate())

spark.sparkContext.setLogLevel("WARN")

print("SparkSession creada")
print(f"Spark Version: {spark.version}")
print(f"Master: {spark.sparkContext.master}")


TALLER EVALUATIVO 1 - INICIO: 2026-06-11 18:49:17


26/06/11 18:49:20 WARN Utils: Your hostname, david-VMware-Virtual-Platform resolves to a loopback address: 127.0.1.1; using 172.16.179.130 instead (on interface ens33)
26/06/11 18:49:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/11 18:49:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession creada
Spark Version: 3.5.8
Master: local[*]


## 2. Celda 2: Actividad 1 - Creación del DataFrame

In [2]:
print("=" * 60)
print("ACTIVIDAD 1: CREACIÓN DEL DATAFRAME Y EXPLORACIÓN")
print("=" * 60)

schema = StructType([
    StructField("transaccion_id",  StringType(),  nullable=False),
    StructField("cliente",         StringType(),  nullable=False),
    StructField("categoria",       StringType(),  nullable=False),
    StructField("producto",        StringType(),  nullable=False),
    StructField("cantidad",        IntegerType(), nullable=False),
    StructField("precio_unitario", DoubleType(),  nullable=False),
    StructField("fecha",           StringType(),  nullable=False),
    StructField("ciudad",          StringType(),  nullable=False),
])

# Datos aumentados a 200+ registros
data = [
    ("T001", "Carlos García", "Electrónica", "Laptop Dell", 1, 850.00, "2024-01-15", "Bogotá"),
    ("T002", "Ana Martínez", "Hogar", "Aspiradora", 2, 120.50, "2024-01-16", "Medellín"),
    ("T003", "Luis Pérez", "Electrónica", "Mouse Inalámbrico", 3, 25.99, "2024-01-17", "Bogotá"),
    ("T004", "María López", "Deportes", "Bicicleta", 1, 450.00, "2024-01-18", "Cali"),
    ("T005", "Carlos García", "Hogar", "Set de Ollas", 1, 89.90, "2024-01-19", "Bogotá"),
    ("T006", "Pedro Ruiz", "Electrónica", "Teclado Mecánico", 2, 75.00, "2024-01-20", "Medellín"),
    ("T007", "Ana Martínez", "Deportes", "Balón Fútbol", 5, 30.00, "2024-01-21", "Cali"),
    ("T008", "Luis Pérez", "Hogar", "Lámpara LED", 4, 15.50, "2024-01-22", "Bogotá"),
    ("T009", "María López", "Electrónica", "Monitor 24\"", 1, 200.00, "2024-01-23", "Medellín"),
    ("T010", "Pedro Ruiz", "Deportes", "Pesas 10kg", 2, 55.00, "2024-01-24", "Cali"),
    ("T011", "Laura Gómez", "Electrónica", "Auriculares", 3, 45.99, "2024-01-25", "Barranquilla"),
    ("T012", "Juan Rodríguez", "Hogar", "Microondas", 1, 350.00, "2024-01-26", "Bogotá"),
    ("T013", "Sofía Fernández", "Deportes", "Cuerda Saltar", 2, 15.00, "2024-01-27", "Medellín"),
    ("T014", "Diego Sánchez", "Electrónica", "Tablet Samsung", 1, 320.00, "2024-01-28", "Cali"),
    ("T015", "Valentina Díaz", "Hogar", "Licuadora", 1, 89.00, "2024-01-29", "Barranquilla"),
    ("T016", "Andrés Romero", "Deportes", "Caminadora", 1, 899.00, "2024-01-30", "Bogotá"),
    ("T017", "Carolina Mora", "Electrónica", "iPhone 15", 1, 1200.00, "2024-01-31", "Medellín"),
    ("T018", "Javier Castillo", "Hogar", "Cafetera", 2, 65.50, "2024-02-01", "Cali"),
    ("T019", "Natalia Ortiz", "Deportes", "Colchoneta Yoga", 3, 35.00, "2024-02-02", "Bogotá"),
    ("T020", "Ricardo Silva", "Electrónica", "Disco Duro 1TB", 2, 85.00, "2024-02-03", "Barranquilla"),
    ("T021", "Patricia Ríos", "Hogar", "Plancha", 1, 45.00, "2024-02-04", "Medellín"),
    ("T022", "Fernando Herrera", "Deportes", "Raqueta Tenis", 1, 120.00, "2024-02-05", "Cali"),
    ("T023", "Daniela Moya", "Electrónica", "Cargador USB", 5, 12.99, "2024-02-06", "Bogotá"),
    ("T024", "Alberto Cruz", "Hogar", "Ventilador", 2, 55.00, "2024-02-07", "Barranquilla"),
    ("T025", "Camila Vargas", "Deportes", "Guantes Boxeo", 2, 40.00, "2024-02-08", "Medellín"),
    ("T026", "Carlos García", "Electrónica", "Monitor 24\"", 1, 200.00, "2024-02-09", "Bogotá"),
    ("T027", "Ana Martínez", "Hogar", "Robot Aspirador", 1, 450.00, "2024-02-10", "Cali"),
    ("T028", "Luis Pérez", "Deportes", "Termo Agua", 3, 25.00, "2024-02-11", "Barranquilla"),
    ("T029", "María López", "Electrónica", "Laptop Dell", 1, 850.00, "2024-02-12", "Bogotá"),
    ("T030", "Pedro Ruiz", "Hogar", "Sartén Antiadherente", 2, 35.00, "2024-02-13", "Medellín"),
    ("T031", "Laura Gómez", "Deportes", "Tenis Running", 1, 95.00, "2024-02-14", "Cali"),
    ("T032", "Juan Rodríguez", "Electrónica", "Mouse Inalámbrico", 3, 25.99, "2024-02-15", "Bogotá"),
    ("T033", "Sofía Fernández", "Hogar", "Set de Ollas", 1, 89.90, "2024-02-16", "Barranquilla"),
    ("T034", "Diego Sánchez", "Deportes", "Bicicleta", 1, 450.00, "2024-02-17", "Medellín"),
    ("T035", "Valentina Díaz", "Electrónica", "Teclado Mecánico", 1, 75.00, "2024-02-18", "Cali"),
    ("T036", "Andrés Romero", "Hogar", "Lámpara LED", 5, 15.50, "2024-02-19", "Bogotá"),
    ("T037", "Carolina Mora", "Deportes", "Balón Fútbol", 4, 30.00, "2024-02-20", "Barranquilla"),
    ("T038", "Javier Castillo", "Electrónica", "Auriculares", 2, 45.99, "2024-02-21", "Medellín"),
    ("T039", "Natalia Ortiz", "Hogar", "Microondas", 1, 350.00, "2024-02-22", "Cali"),
    ("T040", "Ricardo Silva", "Deportes", "Pesas 10kg", 3, 55.00, "2024-02-23", "Bogotá"),
    ("T041", "Patricia Ríos", "Electrónica", "Tablet Samsung", 1, 320.00, "2024-02-24", "Barranquilla"),
    ("T042", "Fernando Herrera", "Hogar", "Licuadora", 2, 89.00, "2024-02-25", "Medellín"),
    ("T043", "Daniela Moya", "Deportes", "Caminadora", 1, 899.00, "2024-02-26", "Cali"),
    ("T044", "Alberto Cruz", "Electrónica", "iPhone 15", 1, 1200.00, "2024-02-27", "Bogotá"),
    ("T045", "Camila Vargas", "Hogar", "Cafetera", 1, 65.50, "2024-02-28", "Barranquilla"),
    ("T046", "Carlos García", "Deportes", "Colchoneta Yoga", 4, 35.00, "2024-02-29", "Medellín"),
    ("T047", "Ana Martínez", "Electrónica", "Disco Duro 1TB", 1, 85.00, "2024-03-01", "Cali"),
    ("T048", "Luis Pérez", "Hogar", "Plancha", 3, 45.00, "2024-03-02", "Bogotá"),
    ("T049", "María López", "Deportes", "Raqueta Tenis", 2, 120.00, "2024-03-03", "Barranquilla"),
    ("T050", "Pedro Ruiz", "Electrónica", "Cargador USB", 6, 12.99, "2024-03-04", "Medellín"),
    ("T051", "Laura Gómez", "Hogar", "Ventilador", 1, 55.00, "2024-03-05", "Cali"),
    ("T052", "Juan Rodríguez", "Deportes", "Guantes Boxeo", 3, 40.00, "2024-03-06", "Bogotá"),
    ("T053", "Sofía Fernández", "Electrónica", "Laptop Dell", 1, 850.00, "2024-03-07", "Barranquilla"),
    ("T054", "Diego Sánchez", "Hogar", "Robot Aspirador", 1, 450.00, "2024-03-08", "Medellín"),
    ("T055", "Valentina Díaz", "Deportes", "Termo Agua", 2, 25.00, "2024-03-09", "Cali"),
    ("T056", "Andrés Romero", "Electrónica", "Monitor 24\"", 2, 200.00, "2024-03-10", "Bogotá"),
    ("T057", "Carolina Mora", "Hogar", "Sartén Antiadherente", 3, 35.00, "2024-03-11", "Barranquilla"),
    ("T058", "Javier Castillo", "Deportes", "Tenis Running", 1, 95.00, "2024-03-12", "Medellín"),
    ("T059", "Natalia Ortiz", "Electrónica", "Mouse Inalámbrico", 5, 25.99, "2024-03-13", "Cali"),
    ("T060", "Ricardo Silva", "Hogar", "Set de Ollas", 2, 89.90, "2024-03-14", "Bogotá"),
    ("T061", "Patricia Ríos", "Deportes", "Bicicleta", 1, 450.00, "2024-03-15", "Barranquilla"),
    ("T062", "Fernando Herrera", "Electrónica", "Teclado Mecánico", 1, 75.00, "2024-03-16", "Medellín"),
    ("T063", "Daniela Moya", "Hogar", "Lámpara LED", 4, 15.50, "2024-03-17", "Cali"),
    ("T064", "Alberto Cruz", "Deportes", "Balón Fútbol", 3, 30.00, "2024-03-18", "Bogotá"),
    ("T065", "Camila Vargas", "Electrónica", "Auriculares", 2, 45.99, "2024-03-19", "Barranquilla"),
    ("T066", "Carlos García", "Hogar", "Microondas", 1, 350.00, "2024-03-20", "Medellín"),
    ("T067", "Ana Martínez", "Deportes", "Pesas 10kg", 2, 55.00, "2024-03-21", "Cali"),
    ("T068", "Luis Pérez", "Electrónica", "Tablet Samsung", 1, 320.00, "2024-03-22", "Bogotá"),
    ("T069", "María López", "Hogar", "Licuadora", 1, 89.00, "2024-03-23", "Barranquilla"),
    ("T070", "Pedro Ruiz", "Deportes", "Caminadora", 1, 899.00, "2024-03-24", "Medellín"),
    ("T071", "Laura Gómez", "Electrónica", "iPhone 15", 1, 1200.00, "2024-03-25", "Cali"),
    ("T072", "Juan Rodríguez", "Hogar", "Cafetera", 2, 65.50, "2024-03-26", "Bogotá"),
    ("T073", "Sofía Fernández", "Deportes", "Colchoneta Yoga", 3, 35.00, "2024-03-27", "Barranquilla"),
    ("T074", "Diego Sánchez", "Electrónica", "Disco Duro 1TB", 2, 85.00, "2024-03-28", "Medellín"),
    ("T075", "Valentina Díaz", "Hogar", "Plancha", 1, 45.00, "2024-03-29", "Cali"),
    ("T076", "Andrés Romero", "Deportes", "Raqueta Tenis", 1, 120.00, "2024-03-30", "Bogotá"),
    ("T077", "Carolina Mora", "Electrónica", "Cargador USB", 4, 12.99, "2024-03-31", "Barranquilla"),
    ("T078", "Javier Castillo", "Hogar", "Ventilador", 2, 55.00, "2024-04-01", "Medellín"),
    ("T079", "Natalia Ortiz", "Deportes", "Guantes Boxeo", 2, 40.00, "2024-04-02", "Cali"),
    ("T080", "Ricardo Silva", "Electrónica", "Laptop Dell", 1, 850.00, "2024-04-03", "Bogotá"),
    ("T081", "Patricia Ríos", "Hogar", "Robot Aspirador", 1, 450.00, "2024-04-04", "Barranquilla"),
    ("T082", "Fernando Herrera", "Deportes", "Termo Agua", 3, 25.00, "2024-04-05", "Medellín"),
    ("T083", "Daniela Moya", "Electrónica", "Monitor 24\"", 1, 200.00, "2024-04-06", "Cali"),
    ("T084", "Alberto Cruz", "Hogar", "Sartén Antiadherente", 2, 35.00, "2024-04-07", "Bogotá"),
    ("T085", "Camila Vargas", "Deportes", "Tenis Running", 2, 95.00, "2024-04-08", "Barranquilla"),
    ("T086", "Carlos García", "Electrónica", "Mouse Inalámbrico", 4, 25.99, "2024-04-09", "Medellín"),
    ("T087", "Ana Martínez", "Hogar", "Set de Ollas", 1, 89.90, "2024-04-10", "Cali"),
    ("T088", "Luis Pérez", "Deportes", "Bicicleta", 1, 450.00, "2024-04-11", "Bogotá"),
    ("T089", "María López", "Electrónica", "Teclado Mecánico", 3, 75.00, "2024-04-12", "Barranquilla"),
    ("T090", "Pedro Ruiz", "Hogar", "Lámpara LED", 6, 15.50, "2024-04-13", "Medellín"),
    ("T091", "Laura Gómez", "Deportes", "Balón Fútbol", 5, 30.00, "2024-04-14", "Cali"),
    ("T092", "Juan Rodríguez", "Electrónica", "Auriculares", 2, 45.99, "2024-04-15", "Bogotá"),
    ("T093", "Sofía Fernández", "Hogar", "Microondas", 1, 350.00, "2024-04-16", "Barranquilla"),
    ("T094", "Diego Sánchez", "Deportes", "Pesas 10kg", 4, 55.00, "2024-04-17", "Medellín"),
    ("T095", "Valentina Díaz", "Electrónica", "Tablet Samsung", 1, 320.00, "2024-04-18", "Cali"),
    ("T096", "Andrés Romero", "Hogar", "Licuadora", 2, 89.00, "2024-04-19", "Bogotá"),
    ("T097", "Carolina Mora", "Deportes", "Caminadora", 1, 899.00, "2024-04-20", "Barranquilla"),
    ("T098", "Javier Castillo", "Electrónica", "iPhone 15", 1, 1200.00, "2024-04-21", "Medellín"),
    ("T099", "Natalia Ortiz", "Hogar", "Cafetera", 2, 65.50, "2024-04-22", "Cali"),
    ("T100", "Ricardo Silva", "Deportes", "Colchoneta Yoga", 4, 35.00, "2024-04-23", "Bogotá"),
    ("T101", "Patricia Ríos", "Electrónica", "Disco Duro 1TB", 1, 85.00, "2024-04-24", "Barranquilla"),
    ("T102", "Fernando Herrera", "Hogar", "Plancha", 3, 45.00, "2024-04-25", "Medellín"),
    ("T103", "Daniela Moya", "Deportes", "Raqueta Tenis", 2, 120.00, "2024-04-26", "Cali"),
    ("T104", "Alberto Cruz", "Electrónica", "Cargador USB", 7, 12.99, "2024-04-27", "Bogotá"),
    ("T105", "Camila Vargas", "Hogar", "Ventilador", 1, 55.00, "2024-04-28", "Barranquilla"),
    ("T106", "Carlos García", "Electrónica", "Laptop Dell", 1, 850.00, "2024-05-01", "Bogotá"),
    ("T107", "Ana Martínez", "Hogar", "Aspiradora", 2, 120.50, "2024-05-02", "Medellín"),
    ("T108", "Luis Pérez", "Electrónica", "Mouse Inalámbrico", 3, 25.99, "2024-05-03", "Bogotá"),
    ("T109", "María López", "Deportes", "Bicicleta", 1, 450.00, "2024-05-04", "Cali"),
    ("T110", "Pedro Ruiz", "Hogar", "Set de Ollas", 1, 89.90, "2024-05-05", "Bogotá"),
    ("T111", "Laura Gómez", "Electrónica", "Teclado Mecánico", 2, 75.00, "2024-05-06", "Medellín"),
    ("T112", "Juan Rodríguez", "Deportes", "Balón Fútbol", 5, 30.00, "2024-05-07", "Cali"),
    ("T113", "Sofía Fernández", "Hogar", "Lámpara LED", 4, 15.50, "2024-05-08", "Bogotá"),
    ("T114", "Diego Sánchez", "Electrónica", "Monitor 24\"", 1, 200.00, "2024-05-09", "Medellín"),
    ("T115", "Valentina Díaz", "Deportes", "Pesas 10kg", 2, 55.00, "2024-05-10", "Cali"),
    ("T116", "Andrés Romero", "Electrónica", "Auriculares", 3, 45.99, "2024-05-11", "Barranquilla"),
    ("T117", "Carolina Mora", "Hogar", "Microondas", 1, 350.00, "2024-05-12", "Bogotá"),
    ("T118", "Javier Castillo", "Deportes", "Cuerda Saltar", 2, 15.00, "2024-05-13", "Medellín"),
    ("T119", "Natalia Ortiz", "Electrónica", "Tablet Samsung", 1, 320.00, "2024-05-14", "Cali"),
    ("T120", "Ricardo Silva", "Hogar", "Licuadora", 1, 89.00, "2024-05-15", "Barranquilla"),
    ("T121", "Patricia Ríos", "Deportes", "Caminadora", 1, 899.00, "2024-05-16", "Bogotá"),
    ("T122", "Fernando Herrera", "Electrónica", "iPhone 15", 1, 1200.00, "2024-05-17", "Medellín"),
    ("T123", "Daniela Moya", "Hogar", "Cafetera", 2, 65.50, "2024-05-18", "Cali"),
    ("T124", "Alberto Cruz", "Deportes", "Colchoneta Yoga", 3, 35.00, "2024-05-19", "Bogotá"),
    ("T125", "Camila Vargas", "Electrónica", "Disco Duro 1TB", 2, 85.00, "2024-05-20", "Barranquilla"),
    ("T126", "Carlos García", "Hogar", "Plancha", 1, 45.00, "2024-05-21", "Medellín"),
    ("T127", "Ana Martínez", "Deportes", "Raqueta Tenis", 1, 120.00, "2024-05-22", "Cali"),
    ("T128", "Luis Pérez", "Electrónica", "Cargador USB", 5, 12.99, "2024-05-23", "Bogotá"),
    ("T129", "María López", "Hogar", "Ventilador", 2, 55.00, "2024-05-24", "Barranquilla"),
    ("T130", "Pedro Ruiz", "Deportes", "Guantes Boxeo", 2, 40.00, "2024-05-25", "Medellín"),
    ("T131", "Laura Gómez", "Electrónica", "Monitor 24\"", 1, 200.00, "2024-05-26", "Bogotá"),
    ("T132", "Juan Rodríguez", "Hogar", "Robot Aspirador", 1, 450.00, "2024-05-27", "Cali"),
    ("T133", "Sofía Fernández", "Deportes", "Termo Agua", 3, 25.00, "2024-05-28", "Barranquilla"),
    ("T134", "Diego Sánchez", "Electrónica", "Laptop Dell", 1, 850.00, "2024-05-29", "Bogotá"),
    ("T135", "Valentina Díaz", "Hogar", "Sartén Antiadherente", 2, 35.00, "2024-05-30", "Medellín"),
    ("T136", "Andrés Romero", "Deportes", "Tenis Running", 1, 95.00, "2024-05-31", "Cali"),
    ("T137", "Carolina Mora", "Electrónica", "Mouse Inalámbrico", 3, 25.99, "2024-06-01", "Bogotá"),
    ("T138", "Javier Castillo", "Hogar", "Set de Ollas", 1, 89.90, "2024-06-02", "Barranquilla"),
    ("T139", "Natalia Ortiz", "Deportes", "Bicicleta", 1, 450.00, "2024-06-03", "Medellín"),
    ("T140", "Ricardo Silva", "Electrónica", "Teclado Mecánico", 1, 75.00, "2024-06-04", "Cali"),
    ("T141", "Patricia Ríos", "Hogar", "Lámpara LED", 5, 15.50, "2024-06-05", "Bogotá"),
    ("T142", "Fernando Herrera", "Deportes", "Balón Fútbol", 4, 30.00, "2024-06-06", "Barranquilla"),
    ("T143", "Daniela Moya", "Electrónica", "Auriculares", 2, 45.99, "2024-06-07", "Medellín"),
    ("T144", "Alberto Cruz", "Hogar", "Microondas", 1, 350.00, "2024-06-08", "Cali"),
    ("T145", "Camila Vargas", "Deportes", "Pesas 10kg", 3, 55.00, "2024-06-09", "Bogotá"),
    ("T146", "Carlos García", "Electrónica", "Tablet Samsung", 1, 320.00, "2024-06-10", "Barranquilla"),
    ("T147", "Ana Martínez", "Hogar", "Licuadora", 2, 89.00, "2024-06-11", "Medellín"),
    ("T148", "Luis Pérez", "Deportes", "Caminadora", 1, 899.00, "2024-06-12", "Cali"),
    ("T149", "María López", "Electrónica", "iPhone 15", 1, 1200.00, "2024-06-13", "Bogotá"),
    ("T150", "Pedro Ruiz", "Hogar", "Cafetera", 1, 65.50, "2024-06-14", "Barranquilla"),
    ("T151", "Laura Gómez", "Deportes", "Colchoneta Yoga", 4, 35.00, "2024-06-15", "Medellín"),
    ("T152", "Juan Rodríguez", "Electrónica", "Disco Duro 1TB", 1, 85.00, "2024-06-16", "Cali"),
    ("T153", "Sofía Fernández", "Hogar", "Plancha", 3, 45.00, "2024-06-17", "Bogotá"),
    ("T154", "Diego Sánchez", "Deportes", "Raqueta Tenis", 2, 120.00, "2024-06-18", "Barranquilla"),
    ("T155", "Valentina Díaz", "Electrónica", "Cargador USB", 6, 12.99, "2024-06-19", "Medellín"),
    ("T156", "Andrés Romero", "Hogar", "Ventilador", 1, 55.00, "2024-06-20", "Cali"),
    ("T157", "Carolina Mora", "Deportes", "Guantes Boxeo", 3, 40.00, "2024-06-21", "Bogotá"),
    ("T158", "Javier Castillo", "Electrónica", "Laptop Dell", 1, 850.00, "2024-06-22", "Barranquilla"),
    ("T159", "Natalia Ortiz", "Hogar", "Robot Aspirador", 1, 450.00, "2024-06-23", "Medellín"),
    ("T160", "Ricardo Silva", "Deportes", "Termo Agua", 2, 25.00, "2024-06-24", "Cali"),
    ("T161", "Patricia Ríos", "Electrónica", "Monitor 24\"", 2, 200.00, "2024-06-25", "Bogotá"),
    ("T162", "Fernando Herrera", "Hogar", "Sartén Antiadherente", 3, 35.00, "2024-06-26", "Barranquilla"),
    ("T163", "Daniela Moya", "Deportes", "Tenis Running", 1, 95.00, "2024-06-27", "Medellín"),
    ("T164", "Alberto Cruz", "Electrónica", "Mouse Inalámbrico", 5, 25.99, "2024-06-28", "Cali"),
    ("T165", "Camila Vargas", "Hogar", "Set de Ollas", 2, 89.90, "2024-06-29", "Bogotá"),
    ("T166", "Carlos García", "Deportes", "Bicicleta", 1, 450.00, "2024-06-30", "Barranquilla"),
    ("T167", "Ana Martínez", "Electrónica", "Teclado Mecánico", 1, 75.00, "2024-07-01", "Medellín"),
    ("T168", "Luis Pérez", "Hogar", "Lámpara LED", 4, 15.50, "2024-07-02", "Cali"),
    ("T169", "María López", "Deportes", "Balón Fútbol", 3, 30.00, "2024-07-03", "Bogotá"),
    ("T170", "Pedro Ruiz", "Electrónica", "Auriculares", 2, 45.99, "2024-07-04", "Barranquilla"),
    ("T171", "Laura Gómez", "Hogar", "Microondas", 1, 350.00, "2024-07-05", "Medellín"),
    ("T172", "Juan Rodríguez", "Deportes", "Pesas 10kg", 2, 55.00, "2024-07-06", "Cali"),
    ("T173", "Sofía Fernández", "Electrónica", "Tablet Samsung", 1, 320.00, "2024-07-07", "Bogotá"),
    ("T174", "Diego Sánchez", "Hogar", "Licuadora", 1, 89.00, "2024-07-08", "Barranquilla"),
    ("T175", "Valentina Díaz", "Deportes", "Caminadora", 1, 899.00, "2024-07-09", "Medellín"),
    ("T176", "Andrés Romero", "Electrónica", "iPhone 15", 1, 1200.00, "2024-07-10", "Cali"),
    ("T177", "Carolina Mora", "Hogar", "Cafetera", 2, 65.50, "2024-07-11", "Bogotá"),
    ("T178", "Javier Castillo", "Deportes", "Colchoneta Yoga", 3, 35.00, "2024-07-12", "Barranquilla"),
    ("T179", "Natalia Ortiz", "Electrónica", "Disco Duro 1TB", 2, 85.00, "2024-07-13", "Medellín"),
    ("T180", "Ricardo Silva", "Hogar", "Plancha", 1, 45.00, "2024-07-14", "Cali"),
    ("T181", "Patricia Ríos", "Deportes", "Raqueta Tenis", 1, 120.00, "2024-07-15", "Bogotá"),
    ("T182", "Fernando Herrera", "Electrónica", "Cargador USB", 4, 12.99, "2024-07-16", "Barranquilla"),
    ("T183", "Daniela Moya", "Hogar", "Ventilador", 2, 55.00, "2024-07-17", "Medellín"),
    ("T184", "Alberto Cruz", "Deportes", "Guantes Boxeo", 2, 40.00, "2024-07-18", "Cali"),
    ("T185", "Camila Vargas", "Electrónica", "Laptop Dell", 1, 850.00, "2024-07-19", "Bogotá"),
    ("T186", "Carlos García", "Hogar", "Robot Aspirador", 1, 450.00, "2024-07-20", "Barranquilla"),
    ("T187", "Ana Martínez", "Deportes", "Termo Agua", 3, 25.00, "2024-07-21", "Medellín"),
    ("T188", "Luis Pérez", "Electrónica", "Monitor 24\"", 1, 200.00, "2024-07-22", "Cali"),
    ("T189", "María López", "Hogar", "Sartén Antiadherente", 2, 35.00, "2024-07-23", "Bogotá"),
    ("T190", "Pedro Ruiz", "Deportes", "Tenis Running", 2, 95.00, "2024-07-24", "Barranquilla"),
    ("T191", "Laura Gómez", "Electrónica", "Mouse Inalámbrico", 4, 25.99, "2024-07-25", "Medellín"),
    ("T192", "Juan Rodríguez", "Hogar", "Set de Ollas", 1, 89.90, "2024-07-26", "Cali"),
    ("T193", "Sofía Fernández", "Deportes", "Bicicleta", 1, 450.00, "2024-07-27", "Bogotá"),
    ("T194", "Diego Sánchez", "Electrónica", "Teclado Mecánico", 3, 75.00, "2024-07-28", "Barranquilla"),
    ("T195", "Valentina Díaz", "Hogar", "Lámpara LED", 6, 15.50, "2024-07-29", "Medellín"),
    ("T196", "Andrés Romero", "Deportes", "Balón Fútbol", 5, 30.00, "2024-07-30", "Cali"),
    ("T197", "Carolina Mora", "Electrónica", "Auriculares", 2, 45.99, "2024-07-31", "Bogotá"),
    ("T198", "Javier Castillo", "Hogar", "Microondas", 1, 350.00, "2024-08-01", "Barranquilla"),
    ("T199", "Natalia Ortiz", "Deportes", "Pesas 10kg", 4, 55.00, "2024-08-02", "Medellín"),
    ("T200", "Ricardo Silva", "Electrónica", "Tablet Samsung", 1, 320.00, "2024-08-03", "Cali"),
    ("T201", "Patricia Ríos", "Hogar", "Licuadora", 2, 89.00, "2024-08-04", "Bogotá"),
    ("T202", "Fernando Herrera", "Deportes", "Caminadora", 1, 899.00, "2024-08-05", "Barranquilla"),
    ("T203", "Daniela Moya", "Electrónica", "iPhone 15", 1, 1200.00, "2024-08-06", "Medellín"),
    ("T204", "Alberto Cruz", "Hogar", "Cafetera", 2, 65.50, "2024-08-07", "Cali"),
    ("T205", "Camila Vargas", "Deportes", "Colchoneta Yoga", 4, 35.00, "2024-08-08", "Bogotá"),
]

df = spark.createDataFrame(data, schema=schema)

print("--- Esquema del DataFrame ---")
df.printSchema()

print("\n--- Primeros 5 registros ---")
df.show(5)

print("\n--- Estadísticas de exploración ---")
total_transacciones = df.count()
categorias_distintas = df.select("categoria").distinct().count()
clientes_distintos = df.select("cliente").distinct().count()

print(f"Total transacciones: {total_transacciones}")
print(f"Categorías distintas: {categorias_distintas}")
print(f"Clientes distintos: {clientes_distintos}")

ACTIVIDAD 1: CREACIÓN DEL DATAFRAME Y EXPLORACIÓN
--- Esquema del DataFrame ---
root
 |-- transaccion_id: string (nullable = false)
 |-- cliente: string (nullable = false)
 |-- categoria: string (nullable = false)
 |-- producto: string (nullable = false)
 |-- cantidad: integer (nullable = false)
 |-- precio_unitario: double (nullable = false)
 |-- fecha: string (nullable = false)
 |-- ciudad: string (nullable = false)


--- Primeros 5 registros ---


+--------------+-------------+-----------+-----------------+--------+---------------+----------+--------+
|transaccion_id|      cliente|  categoria|         producto|cantidad|precio_unitario|     fecha|  ciudad|
+--------------+-------------+-----------+-----------------+--------+---------------+----------+--------+
|          T001|Carlos García|Electrónica|      Laptop Dell|       1|          850.0|2024-01-15|  Bogotá|
|          T002| Ana Martínez|      Hogar|       Aspiradora|       2|          120.5|2024-01-16|Medellín|
|          T003|   Luis Pérez|Electrónica|Mouse Inalámbrico|       3|          25.99|2024-01-17|  Bogotá|
|          T004|  María López|   Deportes|        Bicicleta|       1|          450.0|2024-01-18|    Cali|
|          T005|Carlos García|      Hogar|     Set de Ollas|       1|           89.9|2024-01-19|  Bogotá|
+--------------+-------------+-----------+-----------------+--------+---------------+----------+--------+
only showing top 5 rows


--- Estadísticas de 

## 3. Celda 3: Actividad 2 - Transformaciones

In [3]:
# Imprimimos un encabezado bien visible para separar esta actividad en la salida
print("=" * 60)
print("ACTIVIDAD 2: TRANSFORMACIONES Y CÁLCULOS")
print("=" * 60)

# 1. AGREGAR COLUMNA 'total' (cantidad * precio_unitario)
#    Calcula el subtotal sin descuento por cada producto vendido.
df_transformado = df.withColumn(
    "total",
    col("cantidad") * col("precio_unitario")
)

# 2. AGREGAR COLUMNA 'descuento'
#    Si la cantidad es mayor o igual a 3, se aplica un 10% de descuento sobre el 'total'.
#    En caso contrario, el descuento es 0.0.
df_transformado = df_transformado.withColumn(
    "descuento",
    when(col("cantidad") >= 3, col("total") * 0.10).otherwise(lit(0.0))
)

# 3. AGREGAR COLUMNA 'total_final'
#    Es el total después de restar el descuento aplicado.
df_transformado = df_transformado.withColumn(
    "total_final",
    col("total") - col("descuento")
)

# 4. AGREGAR COLUMNA 'rango_precio'
#    Clasifica cada producto en:
#    - "Bajo":  precio_unitario < 50
#    - "Medio": precio_unitario >= 50 y < 200
#    - "Alto":  precio_unitario >= 200
df_transformado = df_transformado.withColumn(
    "rango_precio",
    when(col("precio_unitario") < 50, "Bajo")
    .when(col("precio_unitario") < 200, "Medio")
    .otherwise("Alto")
)

# Mostramos el DataFrame resultante con todas las nuevas columnas agregadas.
# truncate=False permite ver el contenido completo de cada fila sin cortar.
print("--- DataFrame con transformaciones ---")
df_transformado.show(truncate=False)

ACTIVIDAD 2: TRANSFORMACIONES Y CÁLCULOS
--- DataFrame con transformaciones ---
+--------------+---------------+-----------+-----------------+--------+---------------+----------+------------+------+-----------------+-----------+------------+
|transaccion_id|cliente        |categoria  |producto         |cantidad|precio_unitario|fecha     |ciudad      |total |descuento        |total_final|rango_precio|
+--------------+---------------+-----------+-----------------+--------+---------------+----------+------------+------+-----------------+-----------+------------+
|T001          |Carlos García  |Electrónica|Laptop Dell      |1       |850.0          |2024-01-15|Bogotá      |850.0 |0.0              |850.0      |Alto        |
|T002          |Ana Martínez   |Hogar      |Aspiradora       |2       |120.5          |2024-01-16|Medellín    |241.0 |0.0              |241.0      |Medio       |
|T003          |Luis Pérez     |Electrónica|Mouse Inalámbrico|3       |25.99          |2024-01-17|Bogotá      

## 4. Celda 4: Actividad 3 - Consultas y Agregaciones

In [4]:
print("=" * 60)
print("ACTIVIDAD 3: CONSULTAS Y AGREGACIONES")
print("=" * 60)

# 3.1 TOP 3 TRANSACCIONES POR TOTAL_FINAL
#    Ordena el DataFrame de mayor a menor según el total_final (el valor real pagado después del descuento)
#    y muestra solo las 3 primeras filas, es decir, las transacciones con mayor valor monetario.
print("\n--- 3.1 Top 3 transacciones por total_final ---")
df_transformado.orderBy(col("total_final").desc()).show(3)

# 3.2 TOTAL DE VENTAS POR CATEGORÍA
#    Agrupa los registros por 'categoria' (Electrónica, Hogar, Deportes) y suma los 'total_final' de cada grupo.
#    round(..., 2) redondea el resultado a 2 decimales para mejor legibilidad.
#    alias() renombra la columna resultante como 'ventas_totales'.
print("\n--- 3.2 Total de ventas por categoría ---")
df_transformado.groupBy("categoria").agg(
    round(spark_sum("total_final"), 2).alias("ventas_totales")
).show()

# 3.3 PROMEDIO DE COMPRA POR CLIENTE
#    Agrupa por 'cliente' y calcula el promedio (avg) de 'total_final' para cada uno.
#    Esto indica cuánto gasta en promedio cada cliente por transacción.
#    round(..., 2) redondea a 2 decimales.
print("\n--- 3.3 Promedio de compra por cliente ---")
df_transformado.groupBy("cliente").agg(
    round(avg("total_final"), 2).alias("promedio_compra")
).show()

# 3.4 CANTIDAD DE TRANSACCIONES POR CIUDAD
#    Agrupa por 'ciudad' y cuenta (count) cuántas transacciones (identificadas por 'transaccion_id') hay en cada una.
#    Esto muestra qué ciudades tienen mayor actividad de compras.
print("\n--- 3.4 Cantidad de transacciones por ciudad ---")
df_transformado.groupBy("ciudad").agg(
    count("transaccion_id").alias("num_transacciones")
).show()

# 3.5 CATEGORÍA CON MAYOR VENTA TOTAL
#    Similar al punto 3.2, pero después de sumar las ventas por categoría, ordena de mayor a menor
#    y muestra solo la primera fila (la categoría con mayor facturación total).
print("\n--- 3.5 Categoría con mayor venta total ---")
df_transformado.groupBy("categoria").agg(
    round(spark_sum("total_final"), 2).alias("ventas_totales")
).orderBy(col("ventas_totales").desc()).show(1)

# 3.6 CLIENTES DISTINTOS
#    Muestra la variable 'clientes_distintos' que seguramente fue calculada previamente
#    (por ejemplo, con df.select("cliente").distinct().count())
#    Indica cuántos clientes únicos hay en el conjunto de datos.
print(f"Clientes distintos: {clientes_distintos}")

ACTIVIDAD 3: CONSULTAS Y AGREGACIONES

--- 3.1 Top 3 transacciones por total_final ---
+--------------+----------------+-----------+---------+--------+---------------+----------+--------+------+---------+-----------+------------+
|transaccion_id|         cliente|  categoria| producto|cantidad|precio_unitario|     fecha|  ciudad| total|descuento|total_final|rango_precio|
+--------------+----------------+-----------+---------+--------+---------------+----------+--------+------+---------+-----------+------------+
|          T122|Fernando Herrera|Electrónica|iPhone 15|       1|         1200.0|2024-05-17|Medellín|1200.0|      0.0|     1200.0|        Alto|
|          T017|   Carolina Mora|Electrónica|iPhone 15|       1|         1200.0|2024-01-31|Medellín|1200.0|      0.0|     1200.0|        Alto|
|          T149|     María López|Electrónica|iPhone 15|       1|         1200.0|2024-06-13|  Bogotá|1200.0|      0.0|     1200.0|        Alto|
+--------------+----------------+-----------+---------+

## 5. Celda 5: Actividad 4 - Modificación y Persistencia

In [5]:
print("=" * 60)
print("ACTIVIDAD 4: MODIFICACIÓN Y PERSISTENCIA")
print("=" * 60)

# 4.1 FILTRADO DE DATOS
#    Se filtran solo las transacciones que cumplen DOS condiciones:
#    - categoría = "Electrónica" (solo productos de electrónica)
#    - total_final > 100 (el monto pagado después del descuento supera los 100)
#    Ambas condiciones se combinan con & (AND)
df_electronica = df_transformado.filter(
    (col("categoria") == "Electrónica") & (col("total_final") > 100)
)

# 4.2 CONTAR REGISTROS FILTRADOS
#    Muestra cuántas transacciones cumplen con el filtro anterior
#    Útil para verificar que el filtro funcionó y cuántos datos se procesarán
print(f"Registros filtrados: {df_electronica.count()}")

# 4.3 SELECCIÓN DE COLUMNAS
#    Del DataFrame filtrado, se seleccionan SOLO las columnas necesarias para el resultado final:
#    - transaccion_id: identificador único de la transacción
#    - cliente: nombre del comprador
#    - producto: nombre del producto adquirido
#    - cantidad: número de unidades compradas
#    - total_final: monto pagado (con descuento ya aplicado)
#    - ciudad: ubicación de la compra
df_final = df_electronica.select(
    "transaccion_id",
    "cliente",
    "producto",
    "cantidad",
    "total_final",
    "ciudad"
)

# 4.4 RENOMBRAR COLUMNA
#    La columna 'total_final' se renombra a 'monto_pagado' para que el nombre sea más descriptivo
#    y claro para quien lea el archivo resultante
df_final = df_final.withColumnRenamed("total_final", "monto_pagado")

# 4.5 ORDENAR DATOS
#    Se ordena el DataFrame de MAYOR a MENOR según 'monto_pagado' (descendente)
#    Esto permite que las transacciones más valiosas aparezcan primero en el archivo resultante
df_final = df_final.orderBy(col("monto_pagado").desc())

# 4.6 MOSTRAR VISTA PREVIA
#    Se muestra el DataFrame final antes de persistirlo, para verificar que las transformaciones
#    (filtro, selección, renombres, ordenamiento) se aplicaron correctamente
print("\n--- DataFrame final a persistir ---")
df_final.show()

# 4.7 DEFINIR RUTA DE SALIDA
#    Se construye la ruta completa donde se guardará el archivo usando os.path.expanduser()
#    La tilde (~) se expande automáticamente a la ruta del directorio home del usuario
output_path = os.path.expanduser("~/projects/data/taller1/electronica_filtrado.parquet")

# 4.8 LIMPIAR DIRECTORIO EXISTENTE (OPCIONAL)
#    Si ya existe una carpeta/archivo en esa ruta, se elimina completamente con shutil.rmtree()
#    Esto evita errores al sobrescribir o mezclar datos viejos con nuevos
if os.path.exists(output_path):
    shutil.rmtree(output_path)

# 4.9 PERSISTIR DATOS EN FORMATO PARQUET
#    Se escribe el DataFrame en formato Parquet (formato columnar eficiente para Spark)
#    mode("overwrite"): sobrescribe cualquier dato existente en la ruta
#    .parquet(output_path): especifica la ruta y el formato de salida
df_final.write.mode("overwrite").parquet(output_path)

# 4.10 CONFIRMACIÓN
#     Muestra la ruta donde se guardaron los datos, permitiendo verificar que la persistencia fue exitosa
print(f"\nDatos persistidos en: {output_path}")

ACTIVIDAD 4: MODIFICACIÓN Y PERSISTENCIA
Registros filtrados: 44

--- DataFrame final a persistir ---
+--------------+----------------+--------------+--------+------------+------------+
|transaccion_id|         cliente|      producto|cantidad|monto_pagado|      ciudad|
+--------------+----------------+--------------+--------+------------+------------+
|          T122|Fernando Herrera|     iPhone 15|       1|      1200.0|    Medellín|
|          T017|   Carolina Mora|     iPhone 15|       1|      1200.0|    Medellín|
|          T149|     María López|     iPhone 15|       1|      1200.0|      Bogotá|
|          T044|    Alberto Cruz|     iPhone 15|       1|      1200.0|      Bogotá|
|          T176|   Andrés Romero|     iPhone 15|       1|      1200.0|        Cali|
|          T071|     Laura Gómez|     iPhone 15|       1|      1200.0|        Cali|
|          T203|    Daniela Moya|     iPhone 15|       1|      1200.0|    Medellín|
|          T098| Javier Castillo|     iPhone 15|       1| 

## 6. Celda 6: Verificación de Persistencia

In [6]:
print("=" * 60)
print("VERIFICACIÓN DE PERSISTENCIA")
print("=" * 60)

df_verificacion = spark.read.parquet(output_path)

print("--- Esquema del archivo leído ---")
df_verificacion.printSchema()

print("\n--- Datos leídos del Parquet ---")
df_verificacion.show()

print("\n--- Validación de integridad ---")
registros_originales = df_final.count()
registros_leidos = df_verificacion.count()

print(f"Registros antes de escribir: {registros_originales}")
print(f"Registros después de leer:   {registros_leidos}")

if registros_originales == registros_leidos:
    print("INTEGRIDAD VERIFICADA: Los registros coinciden")
else:
    print("ERROR: Los registros NO coinciden")


print(f"Clientes distintos: {clientes_distintos}")

VERIFICACIÓN DE PERSISTENCIA
--- Esquema del archivo leído ---
root
 |-- transaccion_id: string (nullable = true)
 |-- cliente: string (nullable = true)
 |-- producto: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- monto_pagado: double (nullable = true)
 |-- ciudad: string (nullable = true)


--- Datos leídos del Parquet ---
+--------------+----------------+--------------+--------+------------+------------+
|transaccion_id|         cliente|      producto|cantidad|monto_pagado|      ciudad|
+--------------+----------------+--------------+--------+------------+------------+
|          T017|   Carolina Mora|     iPhone 15|       1|      1200.0|    Medellín|
|          T044|    Alberto Cruz|     iPhone 15|       1|      1200.0|      Bogotá|
|          T071|     Laura Gómez|     iPhone 15|       1|      1200.0|        Cali|
|          T098| Javier Castillo|     iPhone 15|       1|      1200.0|    Medellín|
|          T122|Fernando Herrera|     iPhone 15|       1|    

26/06/12 09:55:57 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 53084383 ms exceeds timeout 120000 ms
26/06/12 09:55:57 WARN SparkContext: Killing executors is not supported by current scheduler.
26/06/12 09:55:58 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint